# DermoGraph-XAI — ResNet50 Baseline
**Team 8 | VIT Bhopal | Skin Lesion Classification**

| | |
|---|---|
| Model | ResNet50 (ImageNet pretrained) |
| Dataset | ALL 6 datasets — HAM10000 + PAD-UFES + ISIC2020 + Melanoma + MIDAS |
| Classes | 7 (Melanoma, Nevi, BCC, AK, BKL, DF, Vasc) |
| Loss | Weighted Cross Entropy |
| Optimizer | AdamW + CosineAnnealingLR |


## Cell 1 — Install & Verify GPU

In [ ]:
!pip install -q timm albumentations

import torch
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")
print("✓ GPU ready" if torch.cuda.is_available() else "✗ NO GPU")


## Cell 2 — Imports & Config

In [ ]:
import os, json, time, warnings, glob, shutil
warnings.filterwarnings('ignore')
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

from sklearn.metrics import (f1_score, roc_auc_score,
                              confusion_matrix, classification_report)

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT      = '/kaggle/working/'
CACHE_DIR   = '/kaggle/working/cache/'
BATCH_SIZE  = 32
N_CLASSES   = 7
N_EPOCHS    = 50
LR          = 1e-4
PATIENCE    = 15
GRAD_ACCUM  = 4
MODEL_NAME  = 'resnet50'
TIMM_STR    = 'resnet50'
MEAN        = [0.485, 0.456, 0.406]
STD         = [0.229, 0.224, 0.225]
CLASS_NAMES = ['Melanoma','Nevi','Basal Cell Carcinoma',
               'Actinic Keratosis','Benign Keratosis',
               'Dermatofibroma','Vascular']

torch.manual_seed(42)
np.random.seed(42)
os.makedirs(OUTPUT,    exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"✓ Config ready | Device: {DEVICE} | Classes: {N_CLASSES} | Model: {MODEL_NAME}")


## Cell 3 — Load ALL 6 Datasets & Remap Paths

In [ ]:
# ── All 6 Kaggle dataset paths ────────────────────────────────────────
HAM1   = '/kaggle/input/datasets/akshat23029/dermograph-ham-images/HAM10000_images_part_1'
HAM2   = '/kaggle/input/datasets/akshat23029/dermograph-ham-images/HAM10000_images_part_2'
PAD    = '/kaggle/input/datasets/akshat23029/dermograph-pad-images/images'
SPLITS = '/kaggle/input/datasets/akshat23029/dermograph-splits'
ISIC   = '/kaggle/input/datasets/akshat23029/dermograph-isic2020'
MELAN  = '/kaggle/input/datasets/akshat23029/dermograph-melanoma-cancer/melanoma_cancer_dataset'
MIDAS  = '/kaggle/input/datasets/akshat23029/dermograph-midas/midasmultimodalimagedatasetforaibasedskincancer'

# Verify all paths
print("Checking dataset paths...")
all_ok = True
for name, path in [
    ('HAM part1',        HAM1),
    ('HAM part2',        HAM2),
    ('PAD images',       PAD),
    ('Splits',           SPLITS),
    ('ISIC 2020',        ISIC),
    ('Melanoma Cancer',  MELAN),
    ('MIDAS',            MIDAS),
]:
    exists = os.path.exists(path)
    n      = len([f for f in os.listdir(path) if not f.startswith('.')]) if exists else 0
    print(f"  {'✓' if exists else '✗'} {name:<20}: {n:,} files")
    if not exists: all_ok = False
assert all_ok, "Missing datasets! Add via + Add Input"

# Load splits
train_df = pd.read_csv(f'{SPLITS}/train.csv')
val_df   = pd.read_csv(f'{SPLITS}/val.csv')
test_df  = pd.read_csv(f'{SPLITS}/test.csv')
with open(f'{SPLITS}/class_weights.json') as f:
    cw_raw = json.load(f)

# Build lookups for datasets needing filename search
pad_lookup   = {}
for part in ['imgs_part_1','imgs_part_2','imgs_part_3']:
    d = f"{PAD}/{part}"
    if not os.path.exists(d): continue
    for fname in os.listdir(d):
        pad_lookup[fname] = f"{d}/{fname}"

isic_lookup  = {}
for fname in os.listdir(ISIC):
    if fname.lower().endswith(('.jpg','.jpeg','.png')):
        isic_lookup[fname] = f"{ISIC}/{fname}"

midas_lookup = {}
for fname in os.listdir(MIDAS):
    if fname.lower().endswith(('.jpg','.jpeg','.png','.JPG','.JPEG')):
        midas_lookup[fname] = f"{MIDAS}/{fname}"

print(f"\nLookup tables built:")
print(f"  PAD-UFES  : {len(pad_lookup):,} images")
print(f"  ISIC 2020 : {len(isic_lookup):,} images")
print(f"  MIDAS     : {len(midas_lookup):,} images")

# Remap Mac paths → Kaggle paths
def remap(mac_path):
    p     = str(mac_path)
    fname = os.path.basename(p)

    if 'HAM10000_images_part_1' in p:
        fp = f"{HAM1}/{fname}"
        return fp if os.path.exists(fp) else None
    if 'HAM10000_images_part_2' in p:
        fp = f"{HAM2}/{fname}"
        return fp if os.path.exists(fp) else None
    if 'PAD-UFES' in p or 'imgs_part' in p:
        return pad_lookup.get(fname)
    if 'ISIC 2020' in p or 'isic2020' in p.lower():
        return isic_lookup.get(fname)
    if 'melanoma_cancer_dataset' in p:
        # folder structure: train/malignant, train/benign, test/malignant, test/benign
        for split in ['train','test']:
            for cls in ['malignant','benign']:
                fp = f"{MELAN}/{split}/{cls}/{fname}"
                if os.path.exists(fp): return fp
        return None
    if 'MIDAS' in p or 'midas' in p.lower():
        return midas_lookup.get(fname)
    return None

print("\nRemapping paths for all 6 datasets...")
for name, df in [('train',train_df),('val',val_df),('test',test_df)]:
    df['kpath'] = df['image_path'].apply(remap)
    df.drop(df[df['kpath'].isna()].index, inplace=True)
    df.drop(df[df['label'] >= N_CLASSES].index, inplace=True)
    df.reset_index(drop=True, inplace=True)
    src = df['source'].value_counts().to_dict()
    print(f"  {name:<6}: {len(df):>6,} images | {src}")

print(f"\n✓ Total training images: {len(train_df):,} (was 9,840 with 2 datasets)")


## Cell 4 — Cache Preprocessed Images (Run Once)

In [ ]:
def remove_hair(img_bgr):
    gray     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (11,11))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask  = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    dk       = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    mask     = cv2.dilate(mask, dk, iterations=1)
    return cv2.inpaint(img_bgr, mask, 3, cv2.INPAINT_TELEA)

all_paths = list(set(
    train_df['kpath'].tolist() +
    val_df['kpath'].tolist() +
    test_df['kpath'].tolist()
))

print(f"Caching {len(all_paths):,} unique images (hair removal + 224x224)...")
print("Runs ONCE — epochs will be ~3-4 min after this")

done = skipped = errors = 0
for src_path in tqdm(all_paths, desc="Preprocessing"):
    fname    = os.path.basename(src_path)
    # Use hash to avoid filename collisions across datasets
    uid      = str(abs(hash(src_path)))[:10]
    dst_path = f"{CACHE_DIR}{uid}_{fname}"
    if os.path.exists(dst_path):
        skipped += 1
        continue
    img = cv2.imread(src_path)
    if img is None:
        errors += 1
        continue
    img = remove_hair(img)
    img = cv2.resize(img, (224,224), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    done += 1

print(f"\n✓ Cache complete: {done:,} processed | {skipped:,} skipped | {errors} errors")

# Build cache lookup and update kpath
cache_lookup = {}
for fname in os.listdir(CACHE_DIR):
    uid = fname[:10]
    cache_lookup[uid] = f"{CACHE_DIR}{fname}"

def to_cache(p):
    uid = str(abs(hash(p)))[:10]
    return cache_lookup.get(uid, p)

for df in [train_df, val_df, test_df]:
    df['kpath'] = df['kpath'].apply(to_cache)

total, used, free = shutil.disk_usage(CACHE_DIR)
print(f"Cache size: {used/1e9:.2f} GB | Free: {free/1e9:.1f} GB")
print("✓ All paths updated to cache")


## Cell 5 — Dataset & Augmentation

In [ ]:
def get_train_transforms():
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.Rotate(limit=180, p=0.7),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3,7), p=0.3),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

class DermDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = cv2.imread(str(row['kpath']))
        if img is None:
            img = np.zeros((224,224,3), dtype=np.uint8)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(int(row['label']), dtype=torch.long)

img, lbl = DermDataset(train_df, get_train_transforms())[0]
print(f"✓ Dataset OK | img: {img.shape} | label: {lbl.item()} = {CLASS_NAMES[lbl.item()]}")


## Cell 6 — DataLoaders & Class Weights

In [ ]:
train_ds = DermDataset(train_df, get_train_transforms())
val_ds   = DermDataset(val_df,   get_val_transforms())
test_ds  = DermDataset(test_df,  get_val_transforms())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

cw_list       = [min(float(cw_raw.get(str(i), 1.0)), 10.0) for i in range(N_CLASSES)]
CLASS_WEIGHTS = torch.tensor(cw_list, dtype=torch.float32).to(DEVICE)

print(f"✓ DataLoaders ready")
print(f"  Train : {len(train_loader):>4} batches ({len(train_ds):,} images)")
print(f"  Val   : {len(val_loader):>4} batches ({len(val_ds):,} images)")
print(f"  Test  : {len(test_loader):>4} batches ({len(test_ds):,} images)")
print(f"\nClass weights (capped at 10.0):")
for i,(name,w) in enumerate(zip(CLASS_NAMES,cw_list)):
    print(f"  {i} {name:<25} {w:.4f}")


## Cell 7 — Model, Loss & Optimizer

In [ ]:
model    = timm.create_model(TIMM_STR, pretrained=True, num_classes=N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)
scaler    = GradScaler()

def criterion(logits, labels):
    return F.cross_entropy(logits, labels, weight=CLASS_WEIGHTS)

def run_eval(model, loader):
    model.eval()
    t_loss = correct = total = 0
    preds_all, labels_all, probs_all = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            probs  = torch.softmax(logits, 1)
            t_loss  += loss.item()
            correct += (logits.argmax(1)==labels).sum().item()
            total   += labels.size(0)
            preds_all.extend(logits.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())
    probs_np = np.array(probs_all, dtype=np.float64)
    probs_np = probs_np / probs_np.sum(axis=1, keepdims=True)
    acc = correct / total
    f1  = f1_score(labels_all, preds_all, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(labels_all, probs_np, multi_class='ovr',
                            average='macro', labels=list(range(N_CLASSES)))
    except: auc = 0.0
    return (t_loss/len(loader), acc, f1, auc,
            np.array(preds_all), np.array(labels_all), probs_np)

print(f"✓ ResNet50 loaded")
print(f"  Parameters : {n_params:.1f}M")
print(f"  Loss       : Weighted CrossEntropy (max_weight=10.0)")
print(f"  Optimizer  : AdamW (lr={LR}, wd=1e-2)")
print(f"  Scheduler  : CosineAnnealingLR (T_max={N_EPOCHS})")


## Cell 8 — Training (Best Model Only — No Disk Waste)

In [ ]:
# Clean any leftover files from previous runs
for f in glob.glob(f'{OUTPUT}{MODEL_NAME}_ep*.pth'):
    os.remove(f)

total, used, free = shutil.disk_usage(OUTPUT)
print(f"Disk before training: {used/1e9:.1f}GB used / {free/1e9:.1f}GB free")

best_acc = best_f1 = best_auc = 0.0
patience = 0
history  = {'epoch':[],'train_loss':[],'val_loss':[],
            'val_acc':[],'val_f1':[],'val_auc':[]}
start    = time.time()

print(f"\nTraining {MODEL_NAME.upper()} — {N_EPOCHS} epochs max, patience={PATIENCE}")
print(f"Saving: {MODEL_NAME}_best.pth only (no per-epoch saves)")
print("="*85)
print(f"{'Ep':>3} | {'TrainLoss':>9} | {'ValLoss':>8} | {'ValAcc':>7} | {'F1':>7} | {'AUC':>7} | {'Min':>5} | Status")
print("="*85)

for epoch in range(N_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Ep {epoch+1:02d}/{N_EPOCHS}", leave=False, ncols=80)
    for step, (imgs, labels) in enumerate(pbar):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step+1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        train_loss += loss.item() * GRAD_ACCUM
        pbar.set_postfix({'loss': f'{loss.item()*GRAD_ACCUM:.3f}'})
    tr_loss = train_loss / len(train_loader)

    # ── Validate ───────────────────────────────────────────────────
    va_loss, va_acc, va_f1, va_auc, _, _, _ = run_eval(model, val_loader)
    scheduler.step()
    elapsed = (time.time()-start)/60
    status  = ''

    if va_acc > best_acc:
        best_acc = va_acc; best_f1 = va_f1; best_auc = va_auc
        patience = 0
        torch.save(model.state_dict(), f'{OUTPUT}{MODEL_NAME}_best.pth')
        status = '⭐ BEST'
    else:
        patience += 1
        status = f'pat {patience}/{PATIENCE}'
        if patience >= PATIENCE: status = '⛔ STOP'

    history['epoch'].append(epoch)
    history['train_loss'].append(round(tr_loss,4))
    history['val_loss'].append(round(va_loss,4))
    history['val_acc'].append(round(va_acc,4))
    history['val_f1'].append(round(va_f1,4))
    history['val_auc'].append(round(va_auc,4))

    print(f"{epoch+1:>3} | {tr_loss:>9.4f} | {va_loss:>8.4f} | {va_acc:>7.4f} | "
          f"{va_f1:>7.4f} | {va_auc:>7.4f} | {elapsed:>5.1f} | {status}")

    with open(f'{OUTPUT}{MODEL_NAME}_history.json','w') as f:
        json.dump(history, f, indent=2)

    if patience >= PATIENCE: break

total_min = (time.time()-start)/60
print("="*85)
print(f"\n✓ Training done in {total_min:.1f} min")
print(f"  Best Val Acc : {best_acc*100:.2f}%")
print(f"  Best Val F1  : {best_f1:.4f}")
print(f"  Best Val AUC : {best_auc:.4f}")
print(f"  Epochs run   : {len(history['epoch'])}")
total,used,free = shutil.disk_usage(OUTPUT)
print(f"  Disk used    : {used/1e9:.1f}GB / {free/1e9:.1f}GB free")


## Cell 9 — Training Curves

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.patch.set_facecolor('#0a0e1a')
for ax in axes:
    ax.set_facecolor('#0f1525')
    ax.spines[['top','right','left','bottom']].set_color('#1e2d4a')
    ax.tick_params(colors='#94a3b8')
    ax.set_xlabel('Epoch', color='#94a3b8')
ep = history['epoch']
axes[0].plot(ep,history['train_loss'],'#FF4444',lw=2,label='Train')
axes[0].plot(ep,history['val_loss'],  '#00e5cc',lw=2,label='Val')
axes[0].set_title('Loss',color='white',fontsize=13,fontweight='bold')
axes[0].legend(facecolor='#0f1525',edgecolor='#1e2d4a',labelcolor='white')
axes[1].plot(ep,history['val_acc'],'#00e5cc',lw=2,label='Val Accuracy')
axes[1].axhline(best_acc,color='#ffc849',ls='--',lw=1.5,label=f'Best {best_acc*100:.2f}%')
axes[1].set_title('Accuracy',color='white',fontsize=13,fontweight='bold')
axes[1].set_ylim(0,1)
axes[1].legend(facecolor='#0f1525',edgecolor='#1e2d4a',labelcolor='white')
axes[2].plot(ep,history['val_f1'], '#a855f7',lw=2,label='F1 Macro')
axes[2].plot(ep,history['val_auc'],'#22c55e',lw=2,label='AUC-ROC')
axes[2].set_title('F1 + AUC',color='white',fontsize=13,fontweight='bold')
axes[2].set_ylim(0,1)
axes[2].legend(facecolor='#0f1525',edgecolor='#1e2d4a',labelcolor='white')
plt.suptitle(f'ResNet50 Training | Best Val={best_acc*100:.2f}% AUC={best_auc:.4f}',
             color='white',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT}{MODEL_NAME}_curves.png',dpi=150,bbox_inches='tight',facecolor='#0a0e1a')
plt.show()
print(f"✓ Saved: {MODEL_NAME}_curves.png")


## Cell 10 — Test Evaluation

In [ ]:
print("Loading best checkpoint...")
model.load_state_dict(torch.load(f'{OUTPUT}{MODEL_NAME}_best.pth', map_location=DEVICE))
_, test_acc, test_f1, test_auc, all_preds, all_labels, all_probs = run_eval(model, test_loader)

print("\n"+"="*55)
print(f"  {MODEL_NAME.upper()} — FINAL TEST RESULTS")
print("="*55)
print(f"  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Test F1 Macro : {test_f1:.4f}")
print(f"  Test AUC-ROC  : {test_auc:.4f}")
print(f"  Best Val Acc  : {best_acc*100:.2f}%")
print(f"  Best Val AUC  : {best_auc:.4f}")
print(f"  Epochs run    : {len(history['epoch'])}")
print("="*55)
present = sorted(set(all_labels))
names   = [CLASS_NAMES[i] for i in present]
print("\nPer-class Report:")
print(classification_report(all_labels,all_preds,labels=present,target_names=names,zero_division=0))


## Cell 11 — Confusion Matrices (%, Counts, TP/FP/FN/TN)

In [ ]:
present = sorted(set(all_labels))
names   = [CLASS_NAMES[i] for i in present]
cm      = confusion_matrix(all_labels, all_preds, labels=present)
cm_pct  = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

# ── Plot 1: Percentage ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11,9))
fig.patch.set_facecolor('#0a0e1a')
sns.heatmap(cm_pct,annot=True,fmt='.1f',cmap='YlOrRd',
            xticklabels=names,yticklabels=names,ax=ax,
            linewidths=0.5,linecolor='#1e2d4a',annot_kws={'size':10,'weight':'bold'})
ax.set_facecolor('#0f1525')
ax.set_xlabel('Predicted',color='white',fontsize=12)
ax.set_ylabel('Actual',color='white',fontsize=12)
ax.set_title(f'{MODEL_NAME.upper()} — Confusion Matrix (%)\nAcc={test_acc*100:.2f}%  F1={test_f1:.4f}  AUC={test_auc:.4f}',
             color='white',fontsize=13,fontweight='bold',pad=15)
plt.setp(ax.get_xticklabels(),rotation=35,ha='right',color='white',fontsize=9)
plt.setp(ax.get_yticklabels(),rotation=0,color='white',fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT}{MODEL_NAME}_cm_pct.png',dpi=150,bbox_inches='tight',facecolor='#0a0e1a')
plt.show(); print("✓ Saved: cm_pct.png")

# ── Plot 2: Count with checkmarks ─────────────────────────────────────
annot = np.empty_like(cm, dtype=object)
for i in range(len(present)):
    for j in range(len(present)):
        c = cm[i,j]; p = cm_pct[i,j]
        annot[i,j] = f'✓{c}\n({p:.1f}%)' if i==j else (f'✗{c}\n({p:.1f}%)' if c>0 else '0')
fig, ax = plt.subplots(figsize=(11,9))
fig.patch.set_facecolor('#0a0e1a')
sns.heatmap(cm,annot=annot,fmt='',cmap='RdYlGn',
            xticklabels=names,yticklabels=names,ax=ax,
            linewidths=0.5,linecolor='#1e2d4a',annot_kws={'size':9,'weight':'bold'})
ax.set_facecolor('#0f1525')
ax.set_xlabel('Predicted',color='white',fontsize=12)
ax.set_ylabel('Actual',color='white',fontsize=12)
ax.set_title(f'{MODEL_NAME.upper()} — Confusion Matrix (Counts)\n✓=Correct  ✗=Wrong  Total={len(all_labels):,}',
             color='white',fontsize=13,fontweight='bold',pad=15)
plt.setp(ax.get_xticklabels(),rotation=35,ha='right',color='white',fontsize=9)
plt.setp(ax.get_yticklabels(),rotation=0,color='white',fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT}{MODEL_NAME}_cm_counts.png',dpi=150,bbox_inches='tight',facecolor='#0a0e1a')
plt.show(); print("✓ Saved: cm_counts.png")

# ── Plot 3: TP/FP/FN/TN quadrants per class ───────────────────────────
TP = cm.diagonal()
FP = cm.sum(axis=0) - TP
FN = cm.sum(axis=1) - TP
TN = cm.sum() - (TP+FP+FN)

import matplotlib.patches as patches
n  = len(present)
cols = 4
rows = (n+cols-1)//cols + 1
fig, axes2 = plt.subplots(rows, cols, figsize=(cols*4, rows*4))
fig.patch.set_facecolor('#0a0e1a')
fig.suptitle(f'{MODEL_NAME.upper()} — TP/FP/FN/TN per Class\nAcc={test_acc*100:.2f}%  F1={test_f1:.4f}  AUC={test_auc:.4f}',
             color='white', fontsize=14, fontweight='bold')
axes2 = axes2.flatten()
QCOLORS = {'TP':'#22c55e','FP':'#ffc849','FN':'#FF4444','TN':'#00e5cc'}

for idx in range(n):
    ax = axes2[idx]
    ax.set_facecolor('#0f1525')
    ax.set_xlim(0,2); ax.set_ylim(0,2)
    ax.set_xticks([0.5,1.5]); ax.set_yticks([0.5,1.5])
    ax.set_xticklabels(['Pred\nPos','Pred\nNeg'],color='white',fontsize=8)
    ax.set_yticklabels(['Act\nPos','Act\nNeg'],color='white',fontsize=8)
    ax.tick_params(colors='white',length=0)
    ax.spines[['top','right','left','bottom']].set_color('#1e2d4a')
    ax.axhline(1,color='#1e2d4a',lw=2); ax.axvline(1,color='#1e2d4a',lw=2)
    for (col,row,lbl,val) in [(0,1,'TP',TP[idx]),(1,1,'FN',FN[idx]),
                               (0,0,'FP',FP[idx]),(1,0,'TN',TN[idx])]:
        rect = patches.FancyBboxPatch((col+0.05,row+0.05),0.90,0.90,
               boxstyle="round,pad=0.02",facecolor=QCOLORS[lbl],alpha=0.85,
               edgecolor='white',linewidth=1.5)
        ax.add_patch(rect)
        ax.text(col+0.5,row+0.60,str(val),ha='center',va='center',
                color='white',fontsize=16,fontweight='bold')
        ax.text(col+0.5,row+0.22,lbl,ha='center',va='center',
                color='white',fontsize=10,fontweight='bold',alpha=0.8)
    sup = int(cm[idx].sum())
    ax.set_title(f'{names[idx]}\n(n={sup})',color='white',fontsize=9,fontweight='bold',pad=6)

# Hide unused subplots, use last for legend
for i in range(n, len(axes2)-1):
    axes2[i].set_visible(False)

leg = axes2[-1]
leg.set_facecolor('#0f1525'); leg.axis('off')
leg.set_title('Legend',color='white',fontsize=11,fontweight='bold')
for i,(lbl,col,desc) in enumerate([
    ('TP','#22c55e','Correctly detected'),
    ('FN','#FF4444','Missed / not detected'),
    ('FP','#ffc849','Wrongly detected'),
    ('TN','#00e5cc','Correctly rejected'),
]):
    y = 0.80 - i*0.20
    r = patches.FancyBboxPatch((0.05,y-0.06),0.15,0.12,
        boxstyle="round,pad=0.01",facecolor=col,alpha=0.85,edgecolor='white',linewidth=1)
    leg.add_patch(r)
    leg.text(0.13,y+0.01,lbl,ha='center',va='center',color='white',fontsize=10,fontweight='bold')
    leg.text(0.28,y+0.01,desc,ha='left',va='center',color='#94a3b8',fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT}{MODEL_NAME}_tp_fp_fn_tn.png',dpi=150,bbox_inches='tight',facecolor='#0a0e1a')
plt.show(); print("✓ Saved: tp_fp_fn_tn.png")

print(f"\n{'Class':<25}{'TP':>5}{'FP':>5}{'FN':>5}{'TN':>6}")
print("-"*45)
for i,name in enumerate(names):
    print(f"{name:<25}{TP[i]:>5}{FP[i]:>5}{FN[i]:>5}{TN[i]:>6}")


## Cell 12 — Per-Class Accuracy

In [ ]:
per_class_acc = cm.diagonal()/cm.sum(axis=1)*100
fig,ax = plt.subplots(figsize=(12,5))
fig.patch.set_facecolor('#0a0e1a'); ax.set_facecolor('#0f1525')
colors = ['#FF4444' if a<70 else '#ffc849' if a<85 else '#22c55e' for a in per_class_acc]
bars = ax.bar(names,per_class_acc,color=colors,edgecolor='white',linewidth=0.5,width=0.6)
ax.axhline(test_acc*100,color='#00e5cc',ls='--',lw=2,label=f'Overall={test_acc*100:.2f}%')
ax.set_title(f'{MODEL_NAME.upper()} — Per-Class Accuracy',color='white',fontsize=13,fontweight='bold')
ax.set_ylabel('Accuracy (%)',color='#94a3b8'); ax.set_ylim(0,115)
ax.tick_params(colors='#94a3b8')
ax.spines[['top','right','left','bottom']].set_color('#1e2d4a')
plt.setp(ax.get_xticklabels(),rotation=35,ha='right',color='white',fontsize=9)
for bar,val in zip(bars,per_class_acc):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1.5,
            f'{val:.1f}%',ha='center',color='white',fontsize=9,fontweight='bold')
patches_list = [mpatches.Patch(color='#22c55e',label='≥85% Good'),
                mpatches.Patch(color='#ffc849',label='70-85% OK'),
                mpatches.Patch(color='#FF4444',label='<70% Poor')]
ax.legend(handles=patches_list+[plt.Line2D([0],[0],color='#00e5cc',ls='--',lw=2,
          label=f'Overall {test_acc*100:.2f}%')],
          facecolor='#0f1525',edgecolor='#1e2d4a',labelcolor='white')
plt.tight_layout()
plt.savefig(f'{OUTPUT}{MODEL_NAME}_per_class_acc.png',dpi=150,bbox_inches='tight',facecolor='#0a0e1a')
plt.show(); print(f"✓ Saved: {MODEL_NAME}_per_class_acc.png")


## Cell 13 — Save Results & Download Reminder

In [ ]:
result = {
    'model':        MODEL_NAME.upper(),
    'type':         'baseline',
    'params_M':     round(n_params,1),
    'dataset':      'ALL 6 datasets',
    'best_val_acc': round(best_acc,4),
    'best_val_f1':  round(best_f1,4),
    'best_val_auc': round(best_auc,4),
    'test_acc':     round(test_acc,4),
    'test_f1':      round(test_f1,4),
    'test_auc':     round(test_auc,4),
    'epochs_run':   len(history['epoch']),
    'history':      history,
}
with open(f'{OUTPUT}{MODEL_NAME}_result.json','w') as f:
    json.dump(result,f,indent=2)

torch.cuda.empty_cache()
total,used,free = shutil.disk_usage(OUTPUT)

print("="*55)
print(f"  ✓ {MODEL_NAME.upper()} COMPLETE")
print("="*55)
print(f"  Test Acc  : {test_acc*100:.2f}%")
print(f"  Test F1   : {test_f1:.4f}")
print(f"  Test AUC  : {test_auc:.4f}")
print(f"  Disk used : {used/1e9:.1f}GB / {free/1e9:.1f}GB free")
print("="*55)
print("\n  ⚠️  SAVE VERSION NOW before closing!")
print("  Click 'Save Version' → 'Save and Run All'")
print("\n  Files to download:")
for fname in sorted(os.listdir(OUTPUT)):
    if MODEL_NAME in fname and not fname.endswith('.ipynb'):
        sz = os.path.getsize(f'{OUTPUT}{fname}')/1024
        print(f"    {fname:<45} {sz:.0f} KB" if sz<1024 else f"    {fname:<45} {sz/1024:.1f} MB")
print(f"\n  ✓ Next model: DenseNet121")
